# Iceberg-to-Iceberg Migration — End-to-End Test

Tests the `s3_to_s3_metadata_migration` DAG with `migration_type=iceberg_to_iceberg`.

The iceberg-to-iceberg strategy handles tables whose data already exists on S3 in Iceberg format
(Parquet files + `metadata/` directory) but are **not registered** in the Hive Metastore (HMS).
The DAG discovers these tables by scanning S3, reads their schema from `metadata.json`, creates
HMS table entries via `CREATE TABLE`, and registers existing data files via the Iceberg `add_files` procedure.

### What this notebook tests

| Table | Partition | Rows | Expected outcome |
|---|---|---|---|
| `test_users` | none | 5 | SUCCESS — basic path |
| `test_events` | `PARTITIONED BY (dt)` identity | 12 (3 parts) | SUCCESS — identity partitions |
| `test_monthly_agg` | `PARTITIONED BY (months(ts))` non-identity | 8 | FAIL at `add_files` — [known limitation](../README.md) |

`test_monthly_agg` is **optional** — set `INCLUDE_FAILURE_CASE = False` in the config cell
to run only the success cases. This is useful when you want a clean DAG run without
partial failures.

### Workflow

1. Run **Step 1** (Setup) to create test data on S3 and upload the Excel config
2. **Trigger the DAG** in Airflow with the parameters printed in Step 2
3. Wait for the DAG run to complete, then run **Step 3** (Verify Initial Load)
4. Run **Step 4** (Add Incremental Data)
5. **Re-trigger the DAG** with the same parameters (Step 5)
6. Run **Step 6** (Verify Incremental Load)
7. Run **Step 7** (Cleanup) when done

### Prerequisites

- Spark session with Iceberg support and S3 access (standard in NX1 JupyterHub)
- The `s3_to_s3_metadata_migration` DAG deployed to Airflow with env files configured:
  - `env.shared` — S3 credentials (`S3_DEST_ENDPOINT`, `S3_DEST_ACCESS_KEY`, `S3_DEST_SECRET_KEY`)
  - `env.migration_dag_metadata` — tracking config (`MIGRATION_TRACKING_DATABASE`, `MIGRATION_TRACKING_LOCATION`)
  - The DAG reads these via `load_dotenv()` at import time; Airflow Variables are **not** used

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
# All paths and names are defined here. Adjust if running on a different tenant.

BUCKET = "nx1poc-pdc-default-ygglold"
TENANT_PREFIX = f"s3a://{BUCKET}/es-tenant-2"
TEST_BASE = f"{TENANT_PREFIX}/test_iceberg_to_iceberg"

WAREHOUSE = f"{TEST_BASE}/warehouse"
EXCEL_S3_PATH = f"{TEST_BASE}/config/test_i2i_migration.xlsx"

TEST_DB = "test_i2i"

# Set to True to include test_monthly_agg — a table with non-identity partitions
# (months(ts)) that will intentionally FAIL at add_files. This exercises the DAG's
# error-handling path but causes the DAG run to show as partially failed.
# Set to False for a clean success-only run.
INCLUDE_FAILURE_CASE = True

TEST_TABLES = ["test_users", "test_events"]
if INCLUDE_FAILURE_CASE:
    TEST_TABLES.append("test_monthly_agg")

# Must match the DAG's env file settings
TRACKING_DB = "migration_tracking"
REPORT_LOCATION = f"s3a://{BUCKET}/es-tenant-2/migration_reports"  # MIGRATION_REPORT_LOCATION

AIRFLOW_URL = "https://airflow-es-tenant-2.nx1poc.nx1cloud.com"
DAG_ID = "s3_to_s3_metadata_migration"

# Expected state after each phase (used by verify cells)
INITIAL_EXPECTED = {
    "test_users":  {"rows": 5,  "partitions": 0, "should_succeed": True},
    "test_events": {"rows": 12, "partitions": 3, "should_succeed": True},
}
INCREMENTAL_EXPECTED = {
    "test_users":  {"rows": 8,  "partitions": 0, "should_succeed": True},
    "test_events": {"rows": 16, "partitions": 4, "should_succeed": True},
}
if INCLUDE_FAILURE_CASE:
    INITIAL_EXPECTED["test_monthly_agg"] = {"rows": 0, "partitions": 0, "should_succeed": False}
    INCREMENTAL_EXPECTED["test_monthly_agg"] = {"rows": 0, "partitions": 0, "should_succeed": False}

print(f"Test tables: {TEST_TABLES}")
print(f"Failure case (test_monthly_agg): {'included' if INCLUDE_FAILURE_CASE else 'excluded'}")

In [ ]:
from pyspark.sql import SparkSession

try:
    _ = spark
    print(f"Using existing Spark session (version {spark.version})")
except NameError:
    spark = SparkSession.builder \
        .appName("test-iceberg-to-iceberg-migration") \
        .enableHiveSupport() \
        .getOrCreate()
    print(f"Created Spark session (version {spark.version})")

spark.sparkContext.setLogLevel("WARN")

# ── S3 helpers ───────────────────────────────────────────────────────────────
from py4j.java_gateway import java_import

java_import(spark._jvm, "org.apache.hadoop.fs.*")


def _fs(path):
    return spark._jvm.org.apache.hadoop.fs.FileSystem.get(
        spark._jvm.java.net.URI(path), spark._jsc.hadoopConfiguration()
    )


def s3_exists(path):
    return _fs(path).exists(spark._jvm.org.apache.hadoop.fs.Path(path))


def s3_delete(path):
    fs = _fs(path)
    p = spark._jvm.org.apache.hadoop.fs.Path(path)
    if fs.exists(p):
        fs.delete(p, True)
        print(f"  Deleted: {path}")
    else:
        print(f"  Not found (skip): {path}")


def s3_write_bytes(path, data):
    fs = _fs(path)
    parent = spark._jvm.org.apache.hadoop.fs.Path(path).getParent()
    if not fs.exists(parent):
        fs.mkdirs(parent)
    out = fs.create(spark._jvm.org.apache.hadoop.fs.Path(path), True)
    out.write(bytearray(data))
    out.close()


def verify_table(full_name, expected, label=""):
    """Check a migrated table against expected values. Returns True if all checks pass."""
    ok = True
    if label:
        print(f"\n── {full_name} ({label}) ──")
    else:
        print(f"\n── {full_name} ──")

    # Existence
    table_exists = False
    try:
        spark.sql(f"DESCRIBE {full_name}")
        table_exists = True
    except Exception:
        pass

    if not expected["should_succeed"]:
        if table_exists:
            n = spark.sql(f"SELECT COUNT(*) c FROM {full_name}").collect()[0]["c"]
            if n == 0:
                print(f"  Table exists, 0 rows — add_files failed as expected  PASS")
            else:
                print(f"  UNEXPECTED: table has {n} rows despite expected add_files failure")
                ok = False
        else:
            print(f"  Table not in HMS — expected (add_files failure path)  PASS")
        return ok

    if not table_exists:
        print(f"  FAIL: table not found in HMS")
        return False

    # Row count
    actual_rows = spark.sql(f"SELECT COUNT(*) c FROM {full_name}").collect()[0]["c"]
    rows_ok = actual_rows == expected["rows"]
    print(f"  Rows:       expected={expected['rows']}, actual={actual_rows}  {'PASS' if rows_ok else 'FAIL'}")
    if not rows_ok:
        ok = False

    # Partitions
    if expected["partitions"] > 0:
        actual_parts = 0
        try:
            actual_parts = spark.sql(f"SELECT * FROM {full_name}.partitions").count()
        except Exception:
            pass
        parts_ok = actual_parts == expected["partitions"]
        print(f"  Partitions: expected={expected['partitions']}, actual={actual_parts}  {'PASS' if parts_ok else 'FAIL'}")
        if not parts_ok:
            ok = False

    # Schema + location
    cols = [(r.col_name, r.data_type) for r in spark.sql(f"DESCRIBE {full_name}").collect()
            if r.col_name and not r.col_name.startswith("#")]
    print(f"  Schema:     {cols}")

    for r in spark.sql(f"DESCRIBE FORMATTED {full_name}").collect():
        if (r.col_name or "").strip() == "Location":
            print(f"  Location:   {(r.data_type or '').strip()}")

    return ok

---
## Step 1: Setup — Prepare Test Data for Initial Load

This section creates three Iceberg tables with test data, then drops the HMS entries
so the tables exist only as data + metadata files on S3. This simulates the real-world
scenario where Iceberg data has been copied to S3 but hasn't been registered in the
destination catalog.

The section is **idempotent** — it cleans up any state from prior runs before creating
fresh test data, so you can re-run it safely.

In [ ]:
# ── Pre-cleanup ──────────────────────────────────────────────────────────────
# Remove any leftover tables and S3 data from prior test runs.

print("Pre-cleanup: removing state from prior runs...\n")

for tbl in TEST_TABLES:
    try:
        spark.sql(f"DROP TABLE IF EXISTS {TEST_DB}.{tbl} PURGE")
        print(f"  Dropped table (if existed): {TEST_DB}.{tbl}")
    except Exception:
        spark.sql(f"DROP TABLE IF EXISTS {TEST_DB}.{tbl}")
        print(f"  Dropped table without purge: {TEST_DB}.{tbl}")

try:
    spark.sql(f"DROP DATABASE IF EXISTS {TEST_DB}")
    print(f"  Dropped database (if existed): {TEST_DB}")
except Exception:
    pass

s3_delete(f"{WAREHOUSE}/{TEST_DB}")
print("\nPre-cleanup done.")

In [ ]:
# ── Create test database and tables ──────────────────────────────────────────

spark.sql(f"CREATE DATABASE IF NOT EXISTS {TEST_DB}")
print(f"Database ready: {TEST_DB}\n")

# ── test_users (unpartitioned) ──────────────────────────────────────────────
users_loc = f"{WAREHOUSE}/{TEST_DB}/test_users"
spark.sql(f"""
    CREATE TABLE {TEST_DB}.test_users (
        id INT, name STRING, email STRING, created_at TIMESTAMP
    ) USING iceberg
    LOCATION '{users_loc}'
    TBLPROPERTIES ('write.format.default' = 'parquet', 'format-version' = '2')
""")
spark.sql(f"""
    INSERT INTO {TEST_DB}.test_users VALUES
    (1, 'Alice',   'alice@test.com',   timestamp '2024-01-15 10:00:00'),
    (2, 'Bob',     'bob@test.com',     timestamp '2024-02-20 11:30:00'),
    (3, 'Charlie', 'charlie@test.com', timestamp '2024-03-10 09:15:00'),
    (4, 'Diana',   'diana@test.com',   timestamp '2024-04-05 14:45:00'),
    (5, 'Eve',     'eve@test.com',     timestamp '2024-05-01 16:00:00')
""")
print("test_users created")

# ── test_events (identity partition on dt) ──────────────────────────────────
events_loc = f"{WAREHOUSE}/{TEST_DB}/test_events"
spark.sql(f"""
    CREATE TABLE {TEST_DB}.test_events (
        id INT, user_id INT, event_type STRING, ts TIMESTAMP, dt STRING
    ) USING iceberg
    PARTITIONED BY (dt)
    LOCATION '{events_loc}'
    TBLPROPERTIES ('write.format.default' = 'parquet', 'format-version' = '2')
""")
spark.sql(f"""
    INSERT INTO {TEST_DB}.test_events VALUES
    (1,  1, 'login',    timestamp '2024-01-01 08:00:00', '2024-01-01'),
    (2,  2, 'login',    timestamp '2024-01-01 08:30:00', '2024-01-01'),
    (3,  1, 'purchase', timestamp '2024-01-01 09:00:00', '2024-01-01'),
    (4,  3, 'login',    timestamp '2024-01-01 10:00:00', '2024-01-01'),
    (5,  1, 'logout',   timestamp '2024-01-02 08:00:00', '2024-01-02'),
    (6,  2, 'purchase', timestamp '2024-01-02 09:30:00', '2024-01-02'),
    (7,  4, 'login',    timestamp '2024-01-02 10:00:00', '2024-01-02'),
    (8,  3, 'logout',   timestamp '2024-01-02 11:00:00', '2024-01-02'),
    (9,  5, 'login',    timestamp '2024-01-03 07:00:00', '2024-01-03'),
    (10, 1, 'login',    timestamp '2024-01-03 08:00:00', '2024-01-03'),
    (11, 2, 'logout',   timestamp '2024-01-03 09:00:00', '2024-01-03'),
    (12, 5, 'purchase', timestamp '2024-01-03 10:00:00', '2024-01-03')
""")
print("test_events created")

# ── test_monthly_agg (non-identity partition: months) ───────────────────────
if INCLUDE_FAILURE_CASE:
    agg_loc = f"{WAREHOUSE}/{TEST_DB}/test_monthly_agg"
    spark.sql(f"""
        CREATE TABLE {TEST_DB}.test_monthly_agg (
            id INT, user_id INT, metric_name STRING, metric_value DOUBLE, ts TIMESTAMP
        ) USING iceberg
        PARTITIONED BY (months(ts))
        LOCATION '{agg_loc}'
        TBLPROPERTIES ('write.format.default' = 'parquet', 'format-version' = '2')
    """)
    spark.sql(f"""
        INSERT INTO {TEST_DB}.test_monthly_agg VALUES
        (1, 1, 'cpu_usage', 75.5,    timestamp '2024-01-15 10:00:00'),
        (2, 1, 'memory',    8192.0,  timestamp '2024-01-20 11:00:00'),
        (3, 2, 'cpu_usage', 82.3,    timestamp '2024-01-25 09:00:00'),
        (4, 2, 'memory',    16384.0, timestamp '2024-01-30 14:00:00'),
        (5, 3, 'cpu_usage', 45.2,    timestamp '2024-02-05 08:00:00'),
        (6, 3, 'memory',    4096.0,  timestamp '2024-02-10 12:00:00'),
        (7, 1, 'disk_io',   120.5,   timestamp '2024-02-15 16:00:00'),
        (8, 2, 'disk_io',   95.8,    timestamp '2024-02-20 10:00:00')
    """)
    print("test_monthly_agg created (non-identity partition — will FAIL at add_files)")
else:
    print("test_monthly_agg skipped (INCLUDE_FAILURE_CASE=False)")

In [ ]:
# ── Verify test data before proceeding ───────────────────────────────────────

n = spark.sql(f"SELECT COUNT(*) c FROM {TEST_DB}.test_users").collect()[0]["c"]
print(f"test_users:       {n} rows  (unpartitioned)")

n = spark.sql(f"SELECT COUNT(*) c FROM {TEST_DB}.test_events").collect()[0]["c"]
p = spark.sql(f"SELECT * FROM {TEST_DB}.test_events.partitions").count()
print(f"test_events:      {n} rows, {p} partitions  (identity on dt)")

if INCLUDE_FAILURE_CASE:
    n = spark.sql(f"SELECT COUNT(*) c FROM {TEST_DB}.test_monthly_agg").collect()[0]["c"]
    p = spark.sql(f"SELECT * FROM {TEST_DB}.test_monthly_agg.partitions").count()
    print(f"test_monthly_agg: {n} rows, {p} partitions  (non-identity: months(ts))")
    print(f"  This table will intentionally FAIL at add_files")

In [ ]:
# ── Drop HMS entries ─────────────────────────────────────────────────────────
# The DAG expects to find Iceberg data on S3 that is NOT registered in HMS.
# DROP TABLE (without PURGE) removes the HMS entry but keeps all data and
# metadata files on S3.

print("Dropping HMS entries (data stays on S3)...\n")

for tbl in TEST_TABLES:
    spark.sql(f"DROP TABLE {TEST_DB}.{tbl}")
    print(f"  Dropped: {TEST_DB}.{tbl}")

spark.sql(f"DROP DATABASE {TEST_DB}")
print(f"  Dropped database: {TEST_DB}")

# Verify data is still on S3
print("\nVerifying S3 data integrity:")
all_intact = True
for tbl in TEST_TABLES:
    tbl_path = f"{WAREHOUSE}/{TEST_DB}/{tbl}"
    has_meta = s3_exists(f"{tbl_path}/metadata")
    has_data = s3_exists(f"{tbl_path}/data")
    status = "OK" if (has_meta and has_data) else "PROBLEM"
    if status != "OK":
        all_intact = False
    print(f"  {tbl}: metadata/={'OK' if has_meta else 'MISSING'}  data/={'OK' if has_data else 'MISSING'}  [{status}]")

if all_intact:
    print("\nAll S3 data intact. Ready for DAG to discover and register.")
else:
    print("\nWARNING: Some data missing. DROP TABLE may have purged files.")
    print("Re-run this entire Setup section to recreate the test data.")

In [ ]:
# ── Generate and upload Excel config ─────────────────────────────────────────
# The DAG reads an Excel file from S3 to determine which databases/tables
# to migrate. For iceberg_to_iceberg, the required columns are:
#   database       — source (and dest) database name
#   table          — table name(s), comma-separated, or * for all
#   dest_s3_prefix — S3 path prefix where table data lives
#
# The DAG will scan {dest_s3_prefix}/{database}/ for subdirectories that
# contain a metadata/ folder, and match them against the table pattern.

from io import BytesIO
import pandas as pd

config_df = pd.DataFrame([{
    "database":       TEST_DB,
    "table":          "*",
    "dest_s3_prefix": WAREHOUSE,
}])

buf = BytesIO()
config_df.to_excel(buf, index=False, engine="openpyxl")
s3_write_bytes(EXCEL_S3_PATH, buf.getvalue())

print(f"Excel config uploaded to:\n  {EXCEL_S3_PATH}\n")
print("Contents:")
print(config_df.to_string(index=False))

---
## Step 2: Trigger the DAG — Initial Load

The test data is ready on S3. Now trigger the DAG in Airflow:

1. Open the Airflow URL shown below
2. Find the **`s3_to_s3_metadata_migration`** DAG
3. Click **Trigger DAG w/ config** (the play button with options)
4. Paste the JSON parameters shown below
5. Click **Trigger**

The DAG will:
- Parse the Excel config and find `database=test_i2i`, `table=*`
- Scan `{dest_s3_prefix}/test_i2i/` on S3 and discover the test tables
- Read `metadata.json` from each to extract schema and partition spec
- Create HMS tables and run `add_files` to register data files
- Validate row counts and schemas

**If `INCLUDE_FAILURE_CASE=True`**: `test_users` and `test_events` succeed,
`test_monthly_agg` fails at `add_files` (non-identity partition). The DAG run
will show as partially failed — this is expected.

**If `INCLUDE_FAILURE_CASE=False`**: All tables should succeed and the DAG run
should complete cleanly.

Wait for all tasks to complete before proceeding to Step 3.

In [ ]:
import json

trigger_conf = {
    "excel_file_path": EXCEL_S3_PATH,
    "migration_type": "iceberg_to_iceberg",
}

print(f"Airflow: {AIRFLOW_URL}")
print(f"DAG:     {DAG_ID}")
print()
print("Trigger parameters (paste into 'Trigger DAG w/ config'):")
print(json.dumps(trigger_conf, indent=2))

---
## Step 3: Verify — Initial Load Results

Run these cells **after the DAG run has completed**.

Expected:
- **test_users** — 5 rows, correct schema, VALIDATED
- **test_events** — 12 rows, 3 partitions, VALIDATED
- **test_monthly_agg** (if included) — FAILED at `add_files`, 0 data rows

In [ ]:
print("=" * 70)
print("VERIFYING INITIAL LOAD RESULTS")
print("=" * 70)

all_pass = True
for tbl, expected in INITIAL_EXPECTED.items():
    passed = verify_table(f"{TEST_DB}.{tbl}", expected, label="initial")
    if not passed:
        all_pass = False

print("\n" + "=" * 70)
print(f"OVERALL: {'ALL CHECKS PASSED' if all_pass else 'SOME CHECKS FAILED'}")
print("=" * 70)

In [ ]:
# ── Tracking table details ──────────────────────────────────────────────────
# The DAG writes run-level and table-level status to Iceberg tracking tables.

try:
    print("Latest migration runs:\n")
    spark.sql(f"""
        SELECT run_id, status, total_tables, successful_tables, failed_tables,
               missing_tables, started_at, completed_at
        FROM {TRACKING_DB}.s3_migration_runs
        ORDER BY started_at DESC LIMIT 5
    """).show(truncate=False)

    latest_run = spark.sql(f"""
        SELECT run_id FROM {TRACKING_DB}.s3_migration_runs
        ORDER BY started_at DESC LIMIT 1
    """).collect()

    if latest_run:
        rid = latest_run[0]["run_id"]
        print(f"Table status for run {rid}:\n")
        spark.sql(f"""
            SELECT source_table, overall_status, table_create_status,
                   validation_status, source_row_count, dest_hive_row_count,
                   row_count_match, schema_match, table_already_existed,
                   SUBSTRING(error_message, 1, 100) AS error_preview
            FROM {TRACKING_DB}.s3_migration_table_status
            WHERE run_id = '{rid}'
            ORDER BY source_table
        """).show(truncate=False)

except Exception as e:
    print(f"Could not query tracking tables: {e}")
    print(f"Check that TRACKING_DB='{TRACKING_DB}' matches the DAG's env file setting.")

In [ ]:
# ── Display the migration HTML report ────────────────────────────────────────
# The DAG writes an HTML report to S3 after each run. This cell reads the
# latest report and renders it inline.

from IPython.display import HTML, display

try:
    latest_run = spark.sql(f"""
        SELECT run_id FROM {TRACKING_DB}.s3_migration_runs
        ORDER BY started_at DESC LIMIT 1
    """).collect()

    if not latest_run:
        print("No migration runs found.")
    else:
        rid = latest_run[0]["run_id"]
        report_path = f"{REPORT_LOCATION}/{rid}_s3_report.html"
        print(f"Reading report: {report_path}\n")

        fs = _fs(report_path)
        p = spark._jvm.org.apache.hadoop.fs.Path(report_path)
        if not fs.exists(p):
            print(f"Report file not found at {report_path}")
            print("The DAG's MIGRATION_REPORT_LOCATION may differ — check REPORT_LOCATION in the config cell.")
        else:
            reader = spark._jvm.java.io.BufferedReader(
                spark._jvm.java.io.InputStreamReader(fs.open(p), "UTF-8")
            )
            lines = []
            line = reader.readLine()
            while line is not None:
                lines.append(line)
                line = reader.readLine()
            reader.close()

            display(HTML("\n".join(lines)))

except Exception as e:
    print(f"Could not load report: {e}")

---
## Step 4: Setup — Prepare Incremental Data

This section writes **additional Parquet files** directly into the existing S3 `data/`
directories, simulating new data being copied from a source bucket. These files sit
alongside the originals but are not tracked by Iceberg metadata.

On the next DAG run, the strategy **drops and recreates** each table, then runs
`add_files` to import **all** data files (original + new) into a fresh Iceberg table.

New data:
- **test_users**: +3 rows (total: 5 → 8)
- **test_events**: +4 rows in a new partition `dt=2024-01-04` (total: 12 → 16, partitions: 3 → 4)

We do **not** add data to `test_monthly_agg` — it should continue to fail at `add_files`.

In [ ]:
# ── Write incremental data as raw Parquet to S3 ─────────────────────────────
# Simulates new data files arriving in the destination S3 bucket (e.g. via
# an S3-to-S3 copy job). The next DAG run will DROP + CREATE + add_files,
# importing ALL files (original + new) into a fresh Iceberg table.
#
# Layout matters for add_files:
#   - Unpartitioned: new files must sit FLAT in data/ — a wrapper subdir
#     would be read as a Hive-style partition and the files inside skipped.
#   - Partitioned: new files go in a Hive-style partition subdir
#     (e.g. dt=2024-01-04/) to match the existing layout.

from datetime import datetime

print("Writing incremental Parquet data to S3...\n")

# test_users: +3 rows (total 5 → 8). Unpartitioned → append flat to data/.
users_data_path = f"{WAREHOUSE}/{TEST_DB}/test_users/data"
new_users = spark.createDataFrame([
    (6, "Frank", "frank@test.com", datetime(2024, 6, 15, 10, 0, 0)),
    (7, "Grace", "grace@test.com", datetime(2024, 7, 20, 11, 30, 0)),
    (8, "Hank",  "hank@test.com",  datetime(2024, 8, 10, 9, 15, 0)),
], ["id", "name", "email", "created_at"])
new_users.write.mode("append").parquet(users_data_path)
print(f"  test_users: +3 rows → {users_data_path}/ (flat)")

# test_events: +4 rows in new partition dt=2024-01-04 (total 12 → 16, parts 3 → 4)
events_data_path = f"{WAREHOUSE}/{TEST_DB}/test_events/data"
new_events = spark.createDataFrame([
    (13, 1, "login",    datetime(2024, 1, 4, 8, 0, 0),  "2024-01-04"),
    (14, 3, "purchase", datetime(2024, 1, 4, 9, 0, 0),  "2024-01-04"),
    (15, 4, "login",    datetime(2024, 1, 4, 10, 0, 0), "2024-01-04"),
    (16, 5, "logout",   datetime(2024, 1, 4, 11, 0, 0), "2024-01-04"),
], ["id", "user_id", "event_type", "ts", "dt"])
new_events.write.parquet(f"{events_data_path}/dt=2024-01-04")
print(f"  test_events: +4 rows → {events_data_path}/dt=2024-01-04/")

print("\nIncremental data written. Ready to re-trigger the DAG.")

---
## Step 5: Trigger the DAG — Incremental Load

Re-trigger the **same DAG with the same parameters**. The DAG discovers the tables
from S3 (via `metadata.json`), **drops** the existing HMS entries, **recreates** each
table, and runs `add_files` to import ALL data files — both the originals from the
initial load and the new Parquet files written in Step 4.

- `test_users` and `test_events` will show `table_already_existed = true` in tracking
  (the table existed before the drop).
- `test_monthly_agg` (if included) still fails at `add_files` (expected).

Wait for all tasks to complete before proceeding to Step 6.

---
## Step 6: Verify — Incremental Load Results

Run after the second DAG run completes.

Expected:
- **test_users** — 8 rows (was 5), `table_already_existed = true`
- **test_events** — 16 rows (was 12), 4 partitions (was 3), `table_already_existed = true`
- **test_monthly_agg** (if included) — still FAILED (expected)

In [ ]:
print("=" * 70)
print("VERIFYING INCREMENTAL LOAD RESULTS")
print("=" * 70)

all_pass = True
for tbl, expected in INCREMENTAL_EXPECTED.items():
    passed = verify_table(f"{TEST_DB}.{tbl}", expected, label="incremental")
    if not passed:
        all_pass = False

print("\n" + "=" * 70)
print(f"OVERALL: {'ALL CHECKS PASSED' if all_pass else 'SOME CHECKS FAILED'}")
print("=" * 70)

In [ ]:
# ── Compare initial vs incremental runs ──────────────────────────────────────

try:
    print("Last 2 migration runs:\n")
    spark.sql(f"""
        SELECT run_id, status, total_tables, successful_tables, failed_tables,
               started_at, completed_at
        FROM {TRACKING_DB}.s3_migration_runs
        ORDER BY started_at DESC LIMIT 2
    """).show(truncate=False)

    runs = spark.sql(f"""
        SELECT run_id FROM {TRACKING_DB}.s3_migration_runs
        ORDER BY started_at DESC LIMIT 2
    """).collect()

    if len(runs) >= 2:
        incr_run, init_run = runs[0]["run_id"], runs[1]["run_id"]
        print(f"Comparing incremental ({incr_run}) vs initial ({init_run}):\n")
        spark.sql(f"""
            SELECT
                inc.source_table,
                init.dest_hive_row_count AS initial_rows,
                inc.dest_hive_row_count  AS incremental_rows,
                inc.table_already_existed AS existed,
                inc.overall_status
            FROM {TRACKING_DB}.s3_migration_table_status inc
            JOIN {TRACKING_DB}.s3_migration_table_status init
              ON inc.source_table = init.source_table
             AND inc.source_database = init.source_database
            WHERE inc.run_id = '{incr_run}' AND init.run_id = '{init_run}'
            ORDER BY inc.source_table
        """).show(truncate=False)
    else:
        print("Need at least 2 runs to compare. Run the DAG twice first.")

except Exception as e:
    print(f"Could not query tracking tables: {e}")
    print(f"Check that TRACKING_DB='{TRACKING_DB}' matches the DAG's env file setting.")

---
## Step 7: Cleanup

Removes all test artifacts:
- HMS tables and database
- All test data, metadata, and config from S3

**Safe to skip** if you want to inspect results further — the next run of Step 1
will clean up automatically before creating fresh data.

In [ ]:
print("Cleaning up all test artifacts...\n")

# Drop HMS tables (PURGE to also remove Iceberg metadata)
for tbl in TEST_TABLES:
    try:
        spark.sql(f"DROP TABLE IF EXISTS {TEST_DB}.{tbl} PURGE")
        print(f"  Dropped table: {TEST_DB}.{tbl}")
    except Exception:
        try:
            spark.sql(f"DROP TABLE IF EXISTS {TEST_DB}.{tbl}")
            print(f"  Dropped table (no purge): {TEST_DB}.{tbl}")
        except Exception as e:
            print(f"  Skip {TEST_DB}.{tbl}: {e}")

try:
    spark.sql(f"DROP DATABASE IF EXISTS {TEST_DB}")
    print(f"  Dropped database: {TEST_DB}")
except Exception as e:
    print(f"  Skip database: {e}")

# Delete all test data from S3 (warehouse + config)
print()
s3_delete(TEST_BASE)

print("\nCleanup complete.")